# QAOAを用いてMax Cust問題を解く

ここでは、量子コンピュータを用いて、Max Cut問題をQAOA（Quantum Approximate Optimization Algorithm：量子近似最適化アルゴリズム）を用いて解きます。

本サンプルはABCI-Q上で動作します。ABCI-Q上のPrefect環境の導入とワークフローの動作方法はUser Guideの[Prefectマニュアル](https://g-quat-abciq.github.io/abciq-docs/ja/apps/prefect_install/)をご覧ください。

本サンプルは「[QAOA徹底解説：Max-Cutをステップ・バイ・ステップで解く](https://www.freequantumcomputing.com/ja/tutorials/qaoa-max-cut-tutorial/)」のチュートリアルを参考に、quri-parts-oqtopusを用いるように変更しPrefectに対応したものです。

### 1. 必要なライブラリのimport

In [ ]:
from prefect import flow, task

import numpy as np
from scipy.optimize import minimize

from quri_parts.circuit import QuantumCircuit, Parameter
from quri_parts_oqtopus.backend import OqtopusConfig, OqtopusSamplingBackend

### 2. 回路の作成

シンプルな４ノードのグラフを定義し、`create_qaoa_circuit`で２つのパラメータを与えるQAOAの回路を構築します。

この回路は`sample`関数にて、quri-parts-oqtopusを使用し、ABCI-Q上で量子コンピュータ（またはシミュレータ）で実行されます。

QAOA回路に渡すパラメータは`scipy`ライブラリの`minimize`関数を使って最適化されます。`objective`関数では、回路実行によって得られたサンプル結果からMax-Cutの期待値を計算し、その値が最大となるようにパラメータ探索を行います。

In [ ]:
# =====================================
#  Max Cut Edges
# =====================================
EDGES = [
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 0),
]

N_QUBITS = 4

# =====================================
#  Max Cut QAOA Circuit
# =====================================
def create_qaoa_circuit(gamma: float, beta: float) -> QuantumCircuit:
    circuit = QuantumCircuit(N_QUBITS)

    for q in range(N_QUBITS):
        circuit.add_H_gate(q)

    for i, j in EDGES:
        circuit.add_CNOT_gate(i, j)
        circuit.add_RZ_gate(j, gamma)
        circuit.add_CNOT_gate(i, j)

    for q in range(N_QUBITS):
        circuit.add_RX_gate(q, 2.0 * beta)

    return circuit

def bitstring_from_int(value: int) -> str:
    return format(value, f"0{N_QUBITS}b")


def maxcut_score(bitstring: str) -> int:
    score = 0

    for i, j in EDGES:
        if bitstring[i] != bitstring[j]:
            score += 1

    return score

def compute_expected_cost(counts: dict[int, int]) -> float:
    total_shots = sum(counts.values())

    if total_shots == 0:
        raise ValueError("No samples were returned.")

    expectation = 0.0

    for state, count in counts.items():
        bitstring = bitstring_from_int(state)
        cost = maxcut_score(bitstring)
        expectation += cost * count

    return -(expectation / total_shots)

@task
def sample(circuit: QuantumCircuit, shots: int) -> dict[int, int]:
    backend = OqtopusSamplingBackend(OqtopusConfig.from_file("oqtopus-dev"))

    job = backend.sample(
        circuit,
        device_id="qiskit_sim",
        shots=shots,
    )

    return dict(job.result().counts)   

def objective(params):
    gamma, beta = params

    circuit = create_qaoa_circuit(gamma, beta)
    
    counts = sample(circuit, 2000)

    expectation = compute_expected_cost(counts)

    return expectation

@task
def optimize_electron():
    constraints = [
        {"type": "ineq", "fun": lambda x: x[0]},              # gamma >= 0
        {"type": "ineq", "fun": lambda x: np.pi - x[0]},      # gamma <= pi
        {"type": "ineq", "fun": lambda x: x[1]},              # beta >= 0
        {"type": "ineq", "fun": lambda x: np.pi / 2 - x[1]},  # beta <= pi/2
    ]

    initial_params = [0.7, 0.4]

    result = minimize(
        objective,
        initial_params,
        method="COBYLA",
        constraints=constraints,
        options={
            "maxiter": 50,
            "disp": True,
        },
    )

    return result

### 3. 最良解による最終サンプリング

`final_sampling`では引数で渡された最適解パラメータを使って、量子コンピュータで最終サンプリングを実行して結果を得ます。
ここでは`@task`を使いPrefectの単独のタスクとして実行します。

In [ ]:
@task
def final_sampling(result):
    gamma_opt, beta_opt = result['x']

    print("\n=== Final Sampling ===")

    best_circuit = create_qaoa_circuit(
        gamma_opt,
        beta_opt,
    )

    counts = sample(best_circuit, 10000)

    return counts  


### 4. Covalentのワークフローの定義と実行

ワークフローを定義して実行します。
このプログラムでPrefectタスクとして定義しているのは、量子回路のパラメータ最適化を行う`optimize_electoin`、最終サンプリング`final_sampling`、および量子コンピュータを使った`sample`になります。

In [ ]:
@flow(log_prints=True)
def main():
    result = optimize_electron()
    counts = final_sampling(result)

    return counts

res = main()
print(res)

### 5. 最終サンプリングの結果の出力

サンプリングの結果をソートして、カウント数の多い順に出力させことで、最も可能性の高い良い解を見つけます。

出力例：
```
0101 : count=2721 cut=4
1010 : count=2627 cut=4
0011 : count=844 cut=2
1001 : count=830 cut=2
・・・
```

In [ ]:
def print_top_samples(counts: dict[int, int], top_k: int = 10) -> None:
    sorted_counts = sorted(
        counts.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    for state, count in sorted_counts[:top_k]:
        bitstring = bitstring_from_int(state)
        score = maxcut_score(bitstring)

        print(
            f"{bitstring} : "
            f"count={count} "
            f"cut={score}"
        )

print_top_samples(res)